# motor-video · Render Video 01 (Short 1080x1920)

Muestra de **aprobacion de estilo**. Todo corre en este runtime (CPU basta; no necesita GPU).

1. **Runtime -> Ejecutar todo** (Run all).
2. Al final se descarga `video-01.mp4` automaticamente (1080x1920, ~50s, con voz).

> Voz `edge-tts` + fondos procedurales con cifras reales del informe (32.4.1). B-roll IA (Wan) es enhancement para v2.

In [ ]:
import os, subprocess, json
from pathlib import Path
%pip install -q edge-tts

WORK = Path('/content/motor-video'); (WORK/'out').mkdir(parents=True, exist_ok=True)
OUT = WORK/'out'

SECCIONES = [
 {'id':'hook',    'dur':5,  'color':'0x0a2a3a','acento':'0x8fe3cf','titulo':'32.4.1',        'sub':'FALLA DE SEGURIDAD'},
 {'id':'problema','dur':9,  'color':'0x2a0a12','acento':'0xff5a5a','titulo':'32.4.1',        'sub':'LA VERSIÓN ACTUAL'},
 {'id':'incognito','dur':9, 'color':'0x141420','acento':'0x9aa4ff','titulo':'MODO INCÓGNITO','sub':'NO AÍSLA SESIONES'},
 {'id':'riesgo',  'dur':9,  'color':'0x2a0a0a','acento':'0xff7a3a','titulo':'RASTREO',       'sub':'SIN TU CONOCIMIENTO'},
 {'id':'solucion','dur':10, 'color':'0x0a2418','acento':'0x4ade80','titulo':'3 AJUSTES',     'sub':'DRAW.IO SEGURO'},
 {'id':'cta',     'dur':8,  'color':'0x0a0a12','acento':'0x8fe3cf','titulo':'PRÓXIMO VIDEO','sub':'EL CÓMO'},
]
VOZ = 'es-ES-ElviraNeural'
TEXTO = ('Descubrí una falla de seguridad en draw.io. Y lo peor es que afecta a la versión 32.4.1, la actual. '
 'En mi auditoría encontré que el modo de incógnito no aísla las sesiones como debería. '
 'Un atacante podría rastrear los diagramas que abres incluso cuando crees que estás protegido. '
 'Pero hay solución. Con tres ajustes de configuración podés cerrar esa puerta y dejar tu draw.io realmente seguro. '
 'Te muestro cómo en el próximo video.')
print('setup OK · objetivo', sum(s['dur'] for s in SECCIONES), 's')

In [ ]:
import asyncio, edge_tts, subprocess
async def _g():
    await edge_tts.Communicate(TEXTO, VOZ).save(str(OUT/'voz.mp3'))
asyncio.run(_g())
V = float(subprocess.run(['ffprobe','-v','error','-show_entries','format=duration','-of','default=nw=1:nk=1',str(OUT/'voz.mp3')],capture_output=True,text=True).stdout.strip())
# LA VOZ MANDA: secciones escaladas para que el total == duración real del audio
peso = sum(s['dur'] for s in SECCIONES)
for s in SECCIONES: s['dur'] = round(s['dur'] * V / peso, 3)
print('voz.mp3:', round(V,2), 's → secciones escaladas a', sum(s['dur'] for s in SECCIONES), 's')

In [ ]:
import os, subprocess
FONTS=['/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf','/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf']
FONT=next((f for f in FONTS if os.path.exists(f)),None)
assert FONT, 'no hay fuente TTF'

def esc(s): return s.replace('\\','\\\\').replace(':','\\:').replace('%','\\%')

def gen_clip(sec):
    W,H,FPS=1080,1920,30; d=sec['dur']
    filt=("drawbox=x=0:y=0:w={W}:h=12:color={a}@0.9:t=fill,"
          "drawbox=x=0:y={Hm}:w={W}:h=12:color={a}@0.9:t=fill,"
          "drawtext=fontfile={F}:text='{t}':fontsize=110:fontcolor=white:borderw=4:bordercolor=black:x=(w-text_w)/2:y=(h-text_h)/2-60,"
          "drawtext=fontfile={F}:text='{s}':fontsize=52:fontcolor={a}:x=(w-text_w)/2:y=(h-text_h)/2+90"
          ).format(W=W,Hm=H-12,a=sec['acento'],F=FONT,t=esc(sec['titulo']),s=esc(sec['sub']))
    out=OUT/f"{sec['id']}.mp4"
    subprocess.run(['ffmpeg','-y','-f','lavfi','-i',f"color=c={sec['color']}:s={W}x{H}:r={FPS}:d={d}",'-vf',filt,'-c:v','libx264','-pix_fmt','yuv420p','-an',str(out)],check=True)
    print('clip', sec['id'])

for s in SECCIONES: gen_clip(s)

In [ ]:
import subprocess
W,H,FPS=1080,1920,30
def fit(src,dur,tmp):
    filt=(f'[0:v]scale={W}:{H}:force_original_aspect_ratio=increase,crop={W}:{H},'
          f'fps={FPS},setsar=1,tpad=stop_mode=clone:stop_duration={dur},trim=duration={dur}[v]')
    subprocess.run(['ffmpeg','-y','-stream_loop','-1','-i',str(src),'-filter_complex',filt,'-map','[v]','-c:v','libx264','-pix_fmt','yuv420p','-an',str(tmp)],check=True)
    return tmp
clips=[fit(OUT/f"{s['id']}.mp4", s['dur'], OUT/f"_f_{s['id']}.mp4") for s in SECCIONES]
lst=OUT/'concat.txt'; lst.write_text(''.join(f"file '{c}'\n" for c in clips))
subprocess.run(['ffmpeg','-y','-f','concat','-safe','0','-i',str(lst),'-c','copy',str(OUT/'video.mp4')],check=True)
subprocess.run(['ffmpeg','-y','-i',str(OUT/'video.mp4'),'-i',str(OUT/'voz.mp3'),'-c:v','copy','-c:a','aac','-b:a','128k','-map','0:v:0','-map','1:a:0','-af','apad','-t',str(V),'-movflags','+faststart',str(OUT/'video-01.mp4')],check=True)
print('video-01.mp4 listo ·', round(V,2), 's · faststart')

In [ ]:
import subprocess, json
p=str(OUT/'video-01.mp4')
info=json.loads(subprocess.run(['ffprobe','-v','error','-show_entries','stream=width,height,codec_type:format=duration','-of','json',p],capture_output=True,text=True).stdout)
dur=float(info['format']['duration']); st={s['codec_type']:s for s in info['streams']}
w,h=st['video']['width'],st['video']['height']; audio='audio' in st
faststart=b'moov' in open(p,'rb').read(262144)   # moov debe estar al principio
sync=abs(dur - V) <= 0.5                          # vídeo == voz
ok=(w,h)==(1080,1920) and dur<=60 and audio and faststart and sync
print(json.dumps({'res':f'{w}x{h}','dur':round(dur,1),'voz':round(V,1),'audio':audio,'faststart':faststart,'sync':sync,'QA':'PASS' if ok else 'FAIL'}))
assert ok, 'QA FAIL'

In [ ]:
from google.colab import files
files.download(str(OUT/'video-01.mp4'))